In [4]:
# Table 3 format (PC ER(4,2)) - rows: A1..A4, Total, Stdev; cols: methods
import numpy as np
import pandas as pd

EP_TARGET = 25000
N_RUNS = 20

ALG_DIRS = {
    "LIO": "er_lio_4_2",
    "EIA(2.0,0.2)": "er_attack_4_2",
    "EIA(1.1,0.9)": "er_mild_attack_4_2",
    "EIA(1.5,0.67)": "er_moderate_attack_4_2",
    "EIA(1.5,1.5)": "er_uniform_attack_4_2",
    "EIA(0.67,1.5)": "er_reverse_attack_4_2",
    "REFiNE(2.0,0.2)": "er_REFiNE_attack_4_2",
    "REFiNE(1.1,0.9)": "er_REFiNE_moderate_attack_4_2",
    "REFiNE(0.67,1.5)": "er_REFiNE_reverse_attack_4_2",
    "REFiNE(1.5,1.5)": "er_REFiNE_uniform_attack_4_2",
}

def read_row_at_episode(csv_path, ep=EP_TARGET):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    hit = df[df["episode"] == ep]
    return (hit.iloc[-1] if len(hit) > 0 else df.iloc[-1])

rows = []

for alg, subdir in ALG_DIRS.items():
    for i in range(1, N_RUNS + 1):
        csv_path = f"lio/results/er{i}/{subdir}/log.csv"
        try:
            r = read_row_at_episode(csv_path, EP_TARGET)

            # per-agent rewards at episode=25000
            A1 = float(r["A1_reward_total"])
            A2 = float(r["A2_reward_total"])
            A3 = float(r["A3_reward_total"])
            A4 = float(r["A4_reward_total"])

            rewards = np.array([A1, A2, A3, A4], dtype=float)
            total = rewards.sum()
            stdev = rewards.std(ddof=0)  # keep consistent with earlier

            rows.append({
                "Alg": alg, "Run": i,
                "A1": A1, "A2": A2, "A3": A3, "A4": A4,
                "Total": total,
                "Stdev": stdev
            })

        except Exception as e:
            print(f"[WARN] skip {alg} run {i}: {e}")

df = pd.DataFrame(rows)

# mean over 20 runs for each method
summary = df.groupby("Alg")[["A1","A2","A3","A4","Total","Stdev"]].mean()

# reorder columns to match paper order
summary = summary.reindex(list(ALG_DIRS.keys()))

# Table 3 layout: rows = A1..A4, Total, Stdev ; cols = methods
table3 = summary.T
table3.index = ["A1","A2","A3","A4","Total","Stdev"]

# optional: rounding like paper (1 decimal)
table3_round = table3.round(1)

display(table3_round)
table3_round.to_csv("table3_ER42_format.csv")
print("Saved -> table3_ER42_format.csv")


Alg,LIO,"EIA(2.0,0.2)","EIA(1.1,0.9)","EIA(1.5,0.67)","EIA(1.5,1.5)","EIA(0.67,1.5)","REFiNE(2.0,0.2)","REFiNE(1.1,0.9)","REFiNE(0.67,1.5)","REFiNE(1.5,1.5)"
A1,44.8,35.4,51.3,44.0,48.4,38.4,65.2,46.6,31.4,39.7
A2,40.3,88.7,49.8,49.9,59.9,32.9,93.2,58.3,22.6,53.9
A3,37.5,44.9,32.4,43.1,25.8,65.2,37.0,45.9,68.4,46.6
A4,61.8,58.7,47.7,49.3,53.7,52.0,21.8,38.0,62.2,46.3
Total,184.3,227.7,181.1,186.3,187.8,188.5,217.2,188.9,184.6,186.6
Stdev,51.7,47.5,51.7,52.0,51.0,53.2,45.6,51.4,52.0,52.3


Saved -> table3_ER42_format.csv
